In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_squared_error

In [ ]:
np.random.seed(42)

n = 500

# Variáveis contínuas
X_cont = np.random.normal(size=(n, 3))

# Variáveis binárias
X_bin = np.random.binomial(1, 0.5, size=(n, 2))

# Juntando tudo
X = np.hstack([X_cont, X_bin])

# Nomes das variáveis
columns = ['x1', 'x2', 'x3', 'x4_bin', 'x5_bin']

X = pd.DataFrame(X, columns=columns)

# Variável dependente contínua
beta = np.array([2.0, -1.5, 0.5, 3.0, -2.0])
y = X.values @ beta + np.random.normal(scale=1.0, size=n)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [ ]:
cont_features = ['x1', 'x2', 'x3']
bin_features = ['x4_bin', 'x5_bin']

preprocess = ColumnTransformer(
    transformers=[
        ('cont', StandardScaler(), cont_features),
        ('bin', 'passthrough', bin_features)
    ]
)


In [ ]:
pipeline = Pipeline(
    steps=[
        ('preprocess', preprocess),
        ('model', RidgeCV(alphas=np.logspace(-3, 3, 20)))
    ]
)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
print(f"Melhor alpha: {pipeline.named_steps['model'].alpha_:.3f}")

In [ ]:
y_pred = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE no teste:", rmse)

In [ ]:
best_model = pipeline.named_steps['model']

coef_names = cont_features + bin_features
coefs = best_model.coef_

coef_df = pd.DataFrame({
    'variavel': coef_names,
    'coeficiente': coefs
})

print(coef_df)

In [ ]:
novo_dado = pd.DataFrame({
    'x1': [0.3],
    'x2': [-1.2],
    'x3': [0.8],
    'x4_bin': [1],
    'x5_bin': [0]
})


In [ ]:
predicao = pipeline.predict(novo_dado)
print("Predição para o novo dado:", predicao[0])
